# MSW recovery-pathway screening model, version 3.1
**Screening LCA + TEA of six management options for 21 Indonesian cities/regencies, baseline year 2025**

Functional unit: management of **1 tonne of mixed MSW as received at the facility gate** in 2025.
Options: SL sanitary landfill + flare, S1 WtE, S2 RDF to cement kiln, S3 AD of food waste,
S4 PHB from landfill gas, S5 integrated RDF + AD. Baselines: open dump (OD) and sanitary landfill (SL).
Indicators: climate change (GWP100), net levelised cost, fossil cumulative energy demand (CED), land take.

The model lives in the `mswpath` package; every number is read from `data/*.csv`
(built by `build_database_v3.py`). This script runs the full analysis for the 21 RIPS locations and writes
`outputs/`. Scenario discovery (which conditions favour which option) is in `run_discovery.py`.

In [1]:
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from mswpath import MSWModel, read_input, PW, NAMES, SCENARIOS

try:
    ROOT = Path(__file__).resolve().parent
except NameError:                                  # notebook / Colab
    ROOT = Path.cwd()
DATA, OUT = ROOT / "data", ROOT / "outputs"
OUT.mkdir(exist_ok=True)

M = MSWModel(DATA)
raw = read_input(DATA / "input_kota_2025_updated.csv")
H = M.load_cities(raw)
M.check_single_projection()                        # stops if a tonnage would be projected twice
print(f"{len(H)} locations loaded")

21 locations loaded


## Central results (all scenarios)

In [2]:
DET, CHAR, BEST = M.run_central()
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
print(CHAR.round(2).to_string())
print("\nBest feasible pathway by carbon value (USD per tonne CO2e), market scenario")
print(BEST[BEST.scenario == "market"].set_index("city").drop(columns=["scenario", "Q"]).to_string())

                        comp_year                        t_status   Q_2025    Q_alt  Q_managed_M1  Q_managed_M2  moisture   LHV  L0_od  L0_sl  wte_kwh  rdf_yield  rdf_ncv  ad_kwh  phb_kg  phb_tpa  phb_cost                     kiln  road_km  gate_LHV  gate_WtE_scale  gate_PSEL_generated gate_PSEL_M1  gate_PSEL_M2  gate_RDF  gate_PHB  landfilled_S5  norm_dom  norm_nd                  glass flags
city                                                                                                                                                                                                                                                                                                                                                                                                        
Kota Padang                  2023          stated_in_source_table   656.02   656.02           NaN        471.15      0.52  7.19  36.62  45.77   359.69       0.26    18.06   66.19    9.95  2382.66      5.97 

## Monte Carlo: probability of being best, per scenario

In [3]:
MC = {}
for name in SCENARIOS:
    MC[name], smp = M.run_mc(name, keep=(name == "market"))
    if name == "market": SAMPLES = smp
MCALL = pd.concat(MC.values(), ignore_index=True)
winners = lambda df: M.winners(df, H.city)
for name, df in MC.items():
    print("\nProbability of being best | scenario:", name); print(winners(df).round(2).to_string())


Probability of being best | scenario: market
pathway                   SL    S1    S2    S3    S4    S5 best     p
city                                                                 
Kota Padang             0.46  0.01  0.01  0.00  0.03  0.49   S5  0.49
Kab. Bogor              0.43  0.00  0.00  0.07  0.09  0.41   SL  0.43
Kab. Serang             0.54  0.01  0.00  0.06  0.06  0.34   SL  0.54
Kota Denpasar           0.43  0.16  0.02  0.00  0.06  0.32   SL  0.43
Kab. Cianjur            0.48  0.02  0.01  0.02  0.05  0.42   SL  0.48
Kab. Indramayu          0.49  0.02  0.01  0.02  0.05  0.42   SL  0.49
Kota Semarang           0.47  0.03  0.01  0.01  0.06  0.42   SL  0.47
Kab. Brebes             0.46  0.01  0.02  0.00  0.03  0.47   S5  0.47
Kab. Nganjuk            0.51  0.03  0.00  0.00  0.04  0.41   SL  0.51
Kab. Pati               0.29  0.00  0.10  0.00  0.02  0.58   S5  0.58
Kab. Magelang           0.49  0.02  0.02  0.00  0.03  0.44   SL  0.49
Kab. Pandeglang         0.58  0.00  0.00  0.

## Uncertainty sources kept apart, and Spearman sensitivity

In [4]:
USPLIT = M.uncertainty_split()
print("\nMedian 90% interval width over locations\n", USPLIT.round(1).to_string())
SENS = M.spearman(SAMPLES)

/home/user/claude/MSW_pathway_model_v3/mswpath/core.py:404: RuntimeWarning: Mean of empty slice
  **{f"p_best_pc{lo}_{hi}": (pick[(P["pc"] >= lo) & (P["pc"] < hi)] == j).mean()
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/user/claude/MSW_pathway_model_v3/mswpath/core.py:404: RuntimeWarning: Mean of empty slice
  **{f"p_best_pc{lo}_{hi}": (pick[(P["pc"] >= lo) & (P["pc"] < hi)] == j).mean()
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/user/claude/MSW_pathway_model_v3/mswpath/core.py:404: RuntimeWarning: Mean of empty slice
  **{f"p_best_pc{lo}_{hi}": (pick[(P["pc"] >= lo) & (P["pc"] < hi)] == j).mean()
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret

/home/user/claude/MSW_pathway_model_v3/mswpath/core.py:404: RuntimeWarning: Mean of empty slice
  **{f"p_best_pc{lo}_{hi}": (pick[(P["pc"] >= lo) & (P["pc"] < hi)] == j).mean()
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/user/claude/MSW_pathway_model_v3/mswpath/core.py:404: RuntimeWarning: Mean of empty slice
  **{f"p_best_pc{lo}_{hi}": (pick[(P["pc"] >= lo) & (P["pc"] < hi)] == j).mean()
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/user/claude/MSW_pathway_model_v3/mswpath/core.py:404: RuntimeWarning: Mean of empty slice
  **{f"p_best_pc{lo}_{hi}": (pick[(P["pc"] >= lo) & (P["pc"] < hi)] == j).mean()
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret


Median 90% interval width over locations
          G_w90                                  C_w90                                 
source    both composition only parameters only  both composition only parameters only
pathway                                                                               
S1       266.6            210.4           137.5  41.0             12.4            39.1
S2       337.6            132.4           309.6  16.3              5.1            15.5
S3       453.5            253.7           386.0  15.2              0.4            15.1
S4       547.0            198.1           500.7  43.3              5.3            43.3
S5       244.9            109.9           220.9  18.9              5.5            17.3
SL       542.7            203.2           494.3  11.3              0.0            11.3


## CED and land take (central values, market case)

In [5]:
d = DET[(DET.scenario == "market") & (DET.baseline == "OD")]
LCI = d.pivot(index="city", columns="pathway", values="CED")[PW].join(
    d.pivot(index="city", columns="pathway", values="LU")[PW], lsuffix="_CED_MJ", rsuffix="_LU_m2")
print(d.groupby("pathway")[["CED", "LU"]].median().reindex(PW).round(3).to_string())

              CED     LU
pathway                 
SL         77.598  0.081
S1      -4195.677  0.019
S2      -3803.915  0.061
S3       -466.096  0.067
S4       -574.761  0.081
S5      -4487.097  0.045


## Checks against measured values and Level II/III parameter coverage

In [6]:
c, P, s, Q, kw = M.setup(0, 1, True); X = M.model(s, P, c, Q)["X"]
assert abs(X["ad_feed"][0] + X["rdf_in_S5"][0] + X["landfilled_S5"][0] - 1) < 1e-9, "S5 mass balance broken"
assert X["rdf_t_S5"][0] <= X["rdf_in_S5"][0] - X["rdf_water_S5"][0] + 1e-12, "more RDF delivered than produced"
W, CARB, iF, LAM = M.W, M.CARB, M.iF, M.LAM
c_food = 1e3 * (1 - W[iF]) * CARB[iF]
c_gas = 1e3 * (1 - W[iF]) * M.REG.central["vs_ts"] * M.REG.central["y_ch4"] * M.RHO_CH4 * 12 / 16 / 0.6
assert c_gas < c_food, "AD yield exceeds the carbon available in food waste"
CHAR["HHV_dry"] = (CHAR.LHV + LAM * CHAR.moisture) / (1 - CHAR.moisture) + LAM * 9 * 0.06
span = lambda v, f: f"{CHAR[v].min():{f}} to {CHAR[v].max():{f}} (median {CHAR[v].median():{f}})"
BENCH = pd.DataFrame({
    "model, 21 locations": [span("moisture", ".2f"), span("LHV", ".1f"), span("HHV_dry", ".1f"), span("L0_sl", ".0f"),
                            span("rdf_yield", ".2f"), span("rdf_ncv", ".1f"), span("wte_kwh", ".0f"), f"{c_gas / c_food:.2f}"],
    "reference": ["0.53 to 0.56 at transfer points (Prabowo et al. 2019); 0.64 to 0.66 at a landfill (Fiki et al. 2022)",
                  "about 5 to 7 (HHV 6.9 to 9.0 measured, Prabowo et al. 2019; 6.5 assumed, Azis et al. 2021)",
                  "15.7 to 19.2, from HHV 6.9 to 9.0 as received at 53 to 56% moisture (Prabowo et al. 2019)",
                  "no Indonesian measurement; reported for transparency",
                  "0.20 to 0.42 (Indonesian RDF plants, GIZ 2023)",
                  "12.6 to 13.8 for biodried whole waste (GIZ 2023); sorted and dried SRF is higher",
                  "632 kWh/t at 35% gross conversion (Azis et al. 2021); model uses 18% net",
                  "must be below 1"]},
    index=["bulk moisture (-)", "LHV as received (MJ/kg)", "HHV of dry matter (MJ/kg)", "L0 at MCF = 1 (kg CH4/t)",
           "RDF yield (t/t MSW)", "RDF heating value (MJ/kg)", "WtE net electricity (kWh/t)",
           "carbon in biogas / carbon in food"])
print(BENCH.to_string())

TERM = pd.read_csv(DATA / "composition_terminal_partition_2025.csv", dtype={"code": str})
FP = pd.read_csv(DATA / "fraction_parameters_L1_L2_L3.csv", dtype={"code": str})
TERM = TERM.merge(FP[["level", "code", "moisture_wet_fraction", "DOC_dry_fraction", "DOCf", "carbon_dry_fraction"]],
                  on=["level", "code"], how="left")
PCOLS = ("moisture_wet_fraction", "DOC_dry_fraction", "DOCf", "carbon_dry_fraction")
COVER = (TERM.assign(**{f"has_{p}": TERM[p].notna() * TERM.percent_wet_mass for p in PCOLS})
         .groupby(["city_name", "stream"])[[f"has_{p}" for p in PCOLS]].sum().round(1))

                                          model, 21 locations                                                                                             reference
bulk moisture (-)                  0.39 to 0.60 (median 0.51)  0.53 to 0.56 at transfer points (Prabowo et al. 2019); 0.64 to 0.66 at a landfill (Fiki et al. 2022)
LHV as received (MJ/kg)               5.0 to 9.5 (median 7.2)            about 5 to 7 (HHV 6.9 to 9.0 measured, Prabowo et al. 2019; 6.5 assumed, Azis et al. 2021)
HHV of dry matter (MJ/kg)          15.3 to 20.3 (median 18.1)             15.7 to 19.2, from HHV 6.9 to 9.0 as received at 53 to 56% moisture (Prabowo et al. 2019)
L0 at MCF = 1 (kg CH4/t)                 34 to 65 (median 45)                                                  no Indonesian measurement; reported for transparency
RDF yield (t/t MSW)                0.17 to 0.38 (median 0.27)                                                        0.20 to 0.42 (Indonesian RDF plants, GIZ 2023)
RDF heating valu

## Save tables and figures

In [7]:
for nm, df, idx in (("01_harmonised_inputs", H.round(4), True), ("02_characterisation_and_gates", CHAR.round(3), True),
                    ("03_deterministic_results", DET.round(3), False), ("04_best_pathway_by_carbon_value", BEST, False),
                    ("05_montecarlo_results", MCALL.round(4), False), ("06_sensitivity_spearman", SENS.round(3), True),
                    ("07_assumption_register_used", M.REG, True), ("08_fraction_properties_used", M.PROP, True),
                    ("09_validation_benchmarks", BENCH, True), ("10_uncertainty_split", USPLIT.round(2), True),
                    ("11_parameter_coverage_L2L3", COVER, True), ("13_ced_landuse_central", LCI.round(4), True),
                    ("14_lcia_factors_used", M.LCIA, True)):
    df.to_csv(OUT / f"{nm}.csv", index=idx)

COL = {"SL": "#8c8c8c", "S1": "#d62728", "S2": "#ff7f0e", "S3": "#2ca02c", "S4": "#9467bd", "S5": "#1f77b4"}
fig, ax = plt.subplots(1, 3, figsize=(15, 6.5), sharey=True)
for a, key, ttl in zip(ax, ("market", "perpres109", "food_separated_at_source"),
                       ("Market case", "Perpres 109/2025 tariff for WtE\nat 1,000 t/day or more",
                        "What if food waste arrives\nalready separated")):
    winners(MC[key])[PW].iloc[::-1].plot.barh(stacked=True, ax=a, color=[COL[k] for k in PW], width=0.8, legend=False)
    a.set_title(ttl, fontsize=10); a.set_xlabel("Probability of being the best feasible pathway"); a.set_xlim(0, 1); a.set_ylabel("")
ax[2].legend([NAMES[k] for k in PW], loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)
fig.tight_layout(); fig.savefig(OUT / "fig_probability_best.png", dpi=170); plt.close(fig)

fig, ax = plt.subplots(1, 2, figsize=(13, 5)); dmc = MC["market"]
for a, bg, ttl in zip(ax, ("OD", "SL"), ("Baseline: open dump", "Baseline: sanitary landfill")):
    for k in PW[(bg == "SL"):]:
        q = dmc[dmc.pathway == k]
        a.scatter(q[f"dC_{bg}_p50"], q[f"dG_{bg}_p50"], s=15 + 70 * q.p_feasible, c=COL[k], alpha=0.75, label=NAMES[k],
                  edgecolor="k", linewidth=0.3)
    a.axhline(0, c="k", lw=0.6); a.axvline(0, c="k", lw=0.6); a.set_title(ttl + " (one dot per location)", fontsize=10)
    a.set_xlabel("Extra cost over baseline, USD per tonne MSW (median)"); a.set_ylabel("GHG avoided, kg CO2e per tonne MSW (median)")
ax[0].legend(frameon=False, fontsize=8); fig.tight_layout(); fig.savefig(OUT / "fig_tradeoff.png", dpi=170); plt.close(fig)

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
for a, nm, ttl in zip(ax, ("G", "C"), ("Life-cycle GHG", "Net levelised cost")):
    t = SENS.xs(nm, axis=1, level=1)[PW]; top = t.max(axis=1).nlargest(12).index[::-1]
    t.loc[top].plot.barh(ax=a, color=[COL[k] for k in PW], width=0.85, legend=False)
    a.set_xlabel("mean |Spearman rho| over 21 locations"); a.set_title(ttl + ": most influential inputs", fontsize=10)
ax[1].legend([NAMES[k] for k in PW], frameon=False, fontsize=8); fig.tight_layout()
fig.savefig(OUT / "fig_sensitivity.png", dpi=170); plt.close(fig)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for a, ind, lab in zip(ax, ("CED", "LU"), ("Fossil CED, GJ per tonne MSW (negative = saving)", "Land take, m2 per tonne MSW")):
    q = MC["market"]
    vals = [q[q.pathway == k][f"{ind}_p50"].to_numpy() / (1e3 if ind == "CED" else 1) for k in PW]
    a.boxplot(vals, vert=False); a.set_yticks(range(1, len(PW) + 1), [NAMES[k] for k in PW])
    a.axvline(0, c="k", lw=0.5); a.set_xlabel(lab); a.set_title(lab.split(",")[0] + " (Monte Carlo medians, 21 locations)", fontsize=10)
fig.tight_layout(); fig.savefig(OUT / "fig_ced_landuse.png", dpi=170); plt.close(fig)

ROB = pd.DataFrame({nm: winners(MC[nm]).best for nm in SCENARIOS}).reindex(H.city)
ROB.to_csv(OUT / "12_most_probable_by_scenario.csv")
print("\nMost probable pathway by scenario\n", ROB.to_string())
print("Done. Files written to", OUT.resolve())

/tmp/ipykernel_1183/2369295627.py:42: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  a.boxplot(vals, vert=False); a.set_yticks(range(1, len(PW) + 1), [NAMES[k] for k in PW])
/tmp/ipykernel_1183/2369295627.py:42: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  a.boxplot(vals, vert=False); a.set_yticks(range(1, len(PW) + 1), [NAMES[k] for k in PW])



Most probable pathway by scenario
                        market perpres109 food_separated_at_source residues_to_open_dump managed_M1_status_index managed_M2_local_first GWP20 moisture_IPCC_default tonnage_overview_projected docf_rubber_0.5
city                                                                                                                                                                                                         
Kota Padang                S5         S5                       S3                    S5                     NaN                     SL    S5                    S5                         S5              S5
Kab. Bogor                 SL         SL                       S3                    S5                      SL                     SL    S5                    S5                         SL              SL
Kab. Serang                SL         S1                       S3                    SL                     NaN                     SL    S5